# M7 on Kaggle: full-pool DA-FedDistill (E3) and FedMD

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Settings:** Internet *On*. Accelerator: *None*. M7 fits 15 clients per window on CPU.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed.
Time is an estimate, not a measurement: the dev run took about 11 min at 4 threads, and M6 took about 10 times longer on the full pool than on dev, which gives about 2 hours. The calibrator fits may scale differently, so allow 3.

**Output:** `/kaggle/working/m7_full_outputs.zip` contains `reports/m7/full/` and the run folder. Download it and bring it back.


In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full


In [ ]:
# 4 threads, matching the dev run and M6, because results reproduce exactly only at the same thread count
!python scripts/m7_distill.py --env kaggle --pool full --set resources.n_jobs=4


In [ ]:
import shutil, glob, os
stage = "/kaggle/working/m7_stage"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree("reports/m7/full", f"{stage}/reports/m7/full")
runs = sorted(glob.glob("/kaggle/working/results/runs/*_m7_distill_full"))
if runs:
    shutil.copytree(runs[-1], f"{stage}/results/runs/{os.path.basename(runs[-1])}")
shutil.make_archive("/kaggle/working/m7_full_outputs", "zip", stage)
shutil.rmtree(stage)
print(open("reports/m7/full/DISTILL.md", encoding="utf-8").read()[:6000])
